In [1]:
import requests 
import numpy as np 
import h5py
import matplotlib.pyplot as plt
import illustris_python as il
import matplotlib.colors as colors

In [37]:
def get(path, params=None):
    # make HTTP GET request to path
    r = requests.get(path, params=params, headers=headers)

    # raise exception if response code is not HTTP SUCCESS (200)
    r.raise_for_status()

    if r.headers['content-type'] == 'application/json':
        return r.json() # parse json responses automatically

    if 'content-disposition' in r.headers:
        filename = r.headers['content-disposition'].split("filename=")[1]
        with open(filename, 'wb') as f:
            f.write(r.content)
        return filename # return the filename string

    return r

In [66]:
def get_ids_snaps(subhalo_id, snap_num):
    url = "http://www.tng-project.org/api/TNG50-1/snapshots/" + str(snap_num) + "/subhalos/" + str(int(subhalo_id))
    print(url)
    sub = get(url)

    #access merger trees
    mpb1 = get(sub['trees']['sublink_mpb'])
    t = h5py.File(mpb1,'r')
    print(t.keys())
    snapnums = t['SnapNum']
    subfindIDs = t['SubfindID']
    lastprog = t['LastProgenitorID']
    subparent = t['SubhaloParent']
    #print(snapnums,subfindIDs)
    print(subparent[:])

    return subfindIDs[:], snapnums[:]#, lastprog[:]

In [58]:
r = get(baseUrl)
names = [sim['name'] for sim in r['simulations']]

In [67]:
get_ids_snaps(532301,99)

http://www.tng-project.org/api/TNG50-1/snapshots/99/subhalos/532301
<KeysViewHDF5 ['DescendantID', 'FirstProgenitorID', 'FirstSubhaloInFOFGroupID', 'GroupBHMass', 'GroupBHMdot', 'GroupCM', 'GroupFirstSub', 'GroupGasMetalFractions', 'GroupGasMetallicity', 'GroupLen', 'GroupLenType', 'GroupMass', 'GroupMassType', 'GroupNsubs', 'GroupPos', 'GroupSFR', 'GroupStarMetalFractions', 'GroupStarMetallicity', 'GroupVel', 'GroupWindMass', 'Group_M_Crit200', 'Group_M_Crit500', 'Group_M_Mean200', 'Group_M_TopHat200', 'Group_R_Crit200', 'Group_R_Crit500', 'Group_R_Mean200', 'Group_R_TopHat200', 'LastProgenitorID', 'MainLeafProgenitorID', 'Mass', 'MassHistory', 'NextProgenitorID', 'NextSubhaloInFOFGroupID', 'NumParticles', 'RootDescendantID', 'SnapNum', 'SubfindID', 'SubhaloBHMass', 'SubhaloBHMdot', 'SubhaloCM', 'SubhaloGasMetalFractions', 'SubhaloGasMetalFractionsHalfRad', 'SubhaloGasMetalFractionsMaxRad', 'SubhaloGasMetalFractionsSfr', 'SubhaloGasMetalFractionsSfrWeighted', 'SubhaloGasMetallicity', 

(array([532301, 530491, 524534, 522802, 517646, 513866, 507039, 506070,
        499856, 494872, 488491, 480512, 477367, 470945, 466699, 460597,
        456530, 452260, 450355, 448448, 444888, 443330, 444070, 440554,
        436200, 433268, 431703, 429494, 426547, 423565, 420835, 417228,
        413571, 412362, 470932, 464351, 460283, 457179, 451897, 449980,
        448033, 442600, 439336, 434067, 430602, 426895, 421071, 411982,
        405378, 400258, 394298, 388254, 382365, 372772, 361610, 354982,
        344306, 335951, 329286, 320120, 318066, 311464, 302847, 295197,
        290523, 298125, 290158, 285723, 339188, 324490, 312125, 304540,
        281179, 276834, 278502, 250312, 262817, 243804, 288351, 270886,
        251702, 235715, 203401, 232376, 193168, 245790, 228402, 200330,
        167162, 167430, 146599, 125749, 105712,  85235, 105519,  56653,
         36307,  46391, 294071], dtype=int32),
 array([99, 98, 97, 96, 95, 94, 93, 92, 91, 90, 89, 88, 87, 86, 85, 84, 83,
        82, 8

In [78]:
def returnSimpleTree(subhalo_id,snapshot_number):
    # load sublink chunk offsets from header of first file
    basePath = "http://www.tng-project.org/api/TNG50-1/output/"
    #NG100-1/output/groups_099/fof_subhalo_tab_099.*.hdf5
    groupFile=basePath+'groups_'+str(snapshot_number)+'/'+'fof_subhalo_tab_'+str(snapshot_number)+'.0.hdf5'
    with h5py.File(groupFile,'r') as f:
        subhaloFileOffsets = f['Header'].attrs['FileOffsets_Subhalo']
        treeFileOffsets = f['Header'].attrs['FileOffsets_SubLink']

    # calculate target group catalog file chunk which contains this id
    subhaloFileOffsets = int(subhalo_id) - subhaloFileOffsets
    fileNum = np.max( np.where(subhaloFileOffsets >= 0) )
    subhaloFile=basePath+'groups_'+str(snapshot_number)+'/'+'groups_'+str(snapshot_number)+'.'+str(fileNum)+'.hdf5'
    subhaloOffset=subhaloFileOffsets[fileNum]

    #finding the right file for this tree and where exactly to look
    with h5py.File(subhaloFile,'r') as groupFile:
        rowNum=groupFile["Offsets"]['Subhalo_SublinkRowNum'][subhaloOffset]
        lastProgId=groupFile["Offsets"]['Subhalo_SublinkLastProgenitorID'][subhaloOffset]
        subhaloId=groupFile["Offsets"]['Subhalo_SublinkSubhaloID'][subhaloOffset]

    treeFileOffsets=int(rowNum)-treeFileOffsets
    treeFileNum=np.max(np.where(treeFileOffsets >= 0))
    treeFile=basePath+'trees/SubLink/tree_extended.'+str(treeFileNum)+'.hdf5'
    rowStart = treeFileOffsets[treeFileNum]

    with h5py.File(treeFile,'r') as rawTree:
        #finding which entries in tree we're interested in
        firstId = rawTree['RootDescendantID'][rowStart]
        rowStart=rowStart+(firstId-subhaloId)
        lastId = rawTree['LastProgenitorID'][rowStart]
        rowEnd=rowStart+lastId-firstId+1

        nFind=rawTree['SubfindID'][rowStart:rowEnd]
        nSnap=rawTree['SnapNum'][rowStart:rowEnd]
        nSub=rawTree['SubhaloID'][rowStart:rowEnd]
        nFirst=rawTree['FirstProgenitorID'][rowStart:rowEnd]
        nNext=rawTree['NextProgenitorID'][rowStart:rowEnd]
        nDesc=rawTree['DescendantID'][rowStart:rowEnd]

    #initialises the tree
    thisTree=-1*np.ones((136,2),dtype=int)
    thisTree[:,0]=np.arange(99,-1,-1)

    #traces the tree back to the latest subhalo in the mpb
    zIndex=np.argwhere((nFind==subhalo_id) & (nSnap==snapshot_number))[0][0]
    thisIndex=zIndex
    if thisIndex!=0:
        descSub=nDesc[zIndex]
        thisSub=nSub[zIndex]
        descIndex=zIndex+descSub-nSub[zIndex]
        while ((nFirst[descIndex]==thisSub) & (nDesc[descIndex]!=-1)): #while the first progentior of each descendant is this subhalo
            descSub=nDesc[descIndex]
            thisSub=nSub[descIndex]
            descIndex=descIndex+descSub-nSub[descIndex]
            thisIndex=descIndex

    thisSnap=nSnap[thisIndex]
    thisFind=nFind[thisIndex]
    thisTree[99-thisSnap,1]=thisFind # records subfind id of first step

    #initialises the list of merging galaxies
    if thisSnap!=99: #if it doesn't reach z=0 records the subhalo it merges with
        descIndex=thisIndex+nDesc[thisIndex]-nSub[thisIndex]
        descSnap=nSnap[descIndex]
        descFind=nFind[descIndex]
        mergerTree=[[descSnap,descIndex]]
    else:
        mergerTree=[]


    while nFirst[thisIndex]!=-1: # goes through main progenitors
        thisIndex=thisIndex+nFirst[thisIndex]-nSub[thisIndex] #index of next main step in main progenitor branch

        thisSnap=nSnap[thisIndex] #snapshot of this main progenitor
        thisFind=nFind[thisIndex] #subfind id of this main progenitor
        thisTree[99-thisSnap,1]=thisFind # records subfind id of this main progenitor

        nextIndex=thisIndex+0 #stupid python objects
        while nNext[nextIndex]!=-1: # goes through merging halos (next progenitors)

            nextIndex=nextIndex+nNext[nextIndex]-nSub[nextIndex] #index of next progenitor

            #records details of these mergers
            mergerSnap=nSnap[nextIndex]
            mergerSub=nFind[nextIndex]
            mergerTree.append([mergerSnap,mergerSub])

    # got to the end of this branch, so must save it
    filled=np.argwhere(thisTree[:,1]!=-1) # finds the snapshots during which the halo is in the tree
    if filled.size==1:
        thisTree=thisTree[filled[0],:]
    elif thisTree[0,1]==-1: # if the tree doesn't make it to z=0
        thisTree=thisTree[filled[0][0]:filled[-1][0]+1,:]
    else:
        thisTree=thisTree[0:filled[-1][0]+1,:] # if it does

    mergerTree=np.array(mergerTree)
    data={"Main":thisTree}
    data['Mergers']=mergerTree
    return data

In [79]:
returnSimpleTree(532301,99)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'http://www.tng-project.org/api/TNG50-1/output/groups_99/fof_subhalo_tab_99.0.hdf5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [41]:
i = names.index('TNG50-1')

sim = get( r['simulations'][i]['url'] )
sim.keys()

dict_keys(['name', 'description', 'name_alt', 'boxsize', 'z_start', 'z_final', 'cosmology', 'omega_0', 'omega_L', 'omega_B', 'hubble', 'physics_model', 'has_cooling', 'has_starformation', 'has_winds', 'has_blackholes', 'mass_gas', 'mass_dm', 'softening_dm_comoving', 'softening_stars_comoving', 'softening_blackholes_comoving', 'softening_gas_comoving', 'softening_dm_max_phys', 'softening_stars_max_phys', 'softening_blackholes_max_phys', 'softening_gas_max_phys', 'softening_gas_factor', 'softening_gas_comoving_min', 'num_dm', 'num_tr_mc', 'num_tr_vel', 'longids', 'is_uniform', 'is_zoom', 'is_subbox', 'num_files_snapshot', 'num_files_groupcat', 'num_files_rockstar', 'num_files_lhalotree', 'num_files_sublink', 'num_files_ctrees', 'filesize_lhalotree', 'filesize_sublink', 'filesize_ctrees', 'filesize_ics', 'filesize_simulation', 'has_fof', 'has_subfind', 'has_rockstar', 'has_lhalotree', 'has_sublink', 'has_ctrees', 'permission_required', 'num_snapshots', 'url', 'parent_simulation', 'child_s

In [17]:
snaps = get( sim['snapshots'] )
print(snaps[-1])
snap = get( snaps[-1]['url'] )

{'number': 99, 'redshift': 2.22044604925031e-16, 'num_groups_subfind': 5688113, 'url': 'http://www.tng-project.org/api/TNG50-1/snapshots/99/'}


In [49]:
# subs = get( snap['subhalos'] )
subs = get( snap['subhalos'])
print(subs['count'])
print([subs['results'][i]['id'] for i in np.arange(441000, 540000,1) ])
len(subs['results'])
#subs['results'][532301]
#print(subs['results'][id])

5688113


IndexError: list index out of range

In [53]:
#subs['id':532301]
sub = get( subs['results'][532301]['url'] )

IndexError: list index out of range

In [15]:
ids = np.array([441709, 474008, 532301])
id = ids[0]
sub = get( subs['results'][id]['url'] )
mpb1 = get( sub['trees']['sublink_mpb'] ) 

IndexError: list index out of range